In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Carrega seu arquivo exato
df = pd.read_csv('campeonato-brasileiro-full.csv', encoding='latin-1')  # <--- MUITO IMPORTANTE: encoding!

# Corrige nomes
df = df.rename(columns={
    'mandante_Placar': 'mandante_placar',
    'visitante_Placar': 'visitante_placar',
    'mandante_Estado': 'mandante_estado',
    'visitante_Estado': 'visitante_estado'
})

df['data'] = pd.to_datetime(df['data'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['data']).sort_values('data').reset_index(drop=True)

# Resultado
df['resultado'] = np.where(df['mandante_placar'] > df['visitante_placar'], 0,
                np.where(df['mandante_placar'] < df['visitante_placar'], 2, 1))

# FEATURE ENGINEERING PERFEITO
teams = set(df['mandante']) | set(df['visitante'])
stats = {team: {'home_games':0, 'home_wins':0, 'home_gf':0, 'away_games':0, 'away_wins':0, 'away_gf':0} 
         for team in teams}

feat_list = []
for i, row in df.iterrows():
    h, a = row['mandante'], row['visitante']
    
    # Antes do jogo
    h_win = stats[h]['home_wins'] / max(stats[h]['home_games'], 1)
    a_win = stats[a]['away_wins'] / max(stats[a]['away_games'], 1)
    h_gf  = stats[h]['home_gf']   / max(stats[h]['home_games'], 1)
    a_gf  = stats[a]['away_gf']   / max(stats[a]['away_games'], 1)
    
    feat_list.append({'h_win': h_win, 'a_win': a_win, 'h_gf': h_gf, 'a_gf': a_gf})
    
    # Depois do jogo — atualização
    stats[h]['home_games'] += 1
    stats[a]['away_games'] += 1
    stats[h]['home_gf'] += row['mandante_placar']
    stats[a]['away_gf'] += row['visitante_placar']
    if row['mandante_placar'] > row['visitante_placar']:
        stats[h]['home_wins'] += 1
    elif row['mandante_placar'] < row['visitante_placar']:
        stats[a]['away_wins'] += 1

features_df = pd.DataFrame(feat_list)
df = pd.concat([df.reset_index(drop=True), features_df], axis=1)

# Substitui só os primeiros jogos (os que realmente são 0)
mean_h_win = df['h_win'].mean()
mean_a_win = df['a_win'].mean()
df['h_win'] = df['h_win'].replace(0, mean_h_win)
df['a_win'] = df['a_win'].replace(0, mean_a_win)

# Split 85/15 cronológico
split = int(len(df) * 0.85)
train = df.iloc[:split]
test  = df.iloc[split:]

X_train = train[['h_win', 'a_win', 'h_gf', 'a_gf']]
X_test  = test[['h_win', 'a_win', 'h_gf', 'a_gf']]
y_train = train['resultado']
y_test  = test['resultado']

model = HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.05, max_depth=8, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"Acurácia: {accuracy_score(y_test, pred):.4f}")
print(f"Baseline: {(y_test == 0).mean():.4f}")
print(classification_report(y_test, pred, target_names=['Mandante', 'Empate', 'Visitante']))